# 00 — Pipeline Logs Exploration

This notebook explores **all** logs, CSVs and reports produced by the Snakemake pipeline.
Files are mounted at `/pipeline-logs` from the host `./pipeline_logs` directory.

All cells are defensive: missing files are reported without raising exceptions.

## Pipeline execution order covered here

1. [Setup & directory overview](#1-setup)
2. [Metadata downloads & merging — HTML reports](#2-metadata)
3. [Database creation & validation](#3-database)
4. [Host resolution & download](#4-host-download)
5. [Host mapping creation](#5-host-mapping)
6. [Host FASTA QC & indexing](#6-host-qc)
7. [Host status report](#7-host-status)
8. [Phage & protein FASTA merging](#8-fasta-merge)
9. [Phage & protein FASTA indexing](#9-fasta-index)

## Versioning & Provenance in PBI

PBI uses **three independent layers** to track identity and provenance. Understanding
this is essential before interpreting any pipeline output.

| Layer | What it is | Where it lives | Example |
|-------|-----------|----------------|--------|
| **Package version** | Semantic version of the `pbi` Python library | `src/pbi/__init__.py` → `__version__` | `0.3.0` |
| **Provider schema profile** | Stable label for a data-source's schema/column contract | `workflow/config/config.yaml` → `public_data_provider.schema_profile` | `phagescope_v1` |
| **Pipeline run provenance** | Exact build metadata for one pipeline execution | `/pipeline-logs/csv/pipeline_run_provenance.json` (artifact) | snapshot date, git ref, etc. |

### Why keep them separate?

- The **package version** follows [SemVer](https://semver.org/) and changes whenever the
  `pbi` API or behaviour changes.
- The **schema profile** (`phagescope_v1`) only changes when the upstream data provider
  (PhageScope) modifies its column layout or semantics. It is deliberately stable — you
  can safely compare runs with the same profile even across different package releases.
  Bump it to `phagescope_v2` only when the schema contract changes.
- The **run provenance** captures the exact snapshot date, git commit, and any
  run-specific metadata for a single pipeline execution. It is the traceability record.

> **Tip:** `notebooks/06_reproducibility.ipynb` goes deeper into reading and validating
> all three layers programmatically.


In [1]:
# ---------------------------------------------------------------------------
# Surface version / provenance information at notebook startup
# ---------------------------------------------------------------------------
import sys
import json
from pathlib import Path

# ── pbi package version ──────────────────────────────────────────────────
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))
try:
    import pbi as _pbi
    print(f'pbi package version : {_pbi.__version__}')
except ImportError:
    print('pbi package not importable from this environment')

# ── pipeline run provenance (if pipeline has been executed) ───────────────
_prov_path = Path('/pipeline-logs/csv/pipeline_run_provenance.json')
if _prov_path.exists():
    _prov = json.loads(_prov_path.read_text())
    print('\nPipeline run provenance:')
    for _k, _v in _prov.items():
        print(f'  {_k:<40} {_v}')
else:
    print('\nNo pipeline_run_provenance.json found — pipeline may not have run yet.')
    print('  (This is normal when the notebook is opened before running the pipeline.)')


pbi package version : 0.3.0

Pipeline run provenance:
  pipeline_run_timestamp                   2026-07-04T15:56:50Z
  provider_name                            PhageScope
  provider_release                         rolling
  provider_snapshot_date                   2026-05-11
  provider_schema_profile                  phagescope_v1
  provider_api_base_url                    https://phageapi.deepomics.org
  provider_provenance_mode                 config_pinned
  pbi_version                              
  git_commit                               
  download_records_count                   229


---
<a id='1-setup'></a>
## 1 — Setup & Directory Overview

In [2]:
from pathlib import Path
import json
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# ---------------------------------------------------------------------------
# Root path — matches PBI_LOGS_DIR env var inside the Docker container
# ---------------------------------------------------------------------------
logs_root = Path('/pipeline-logs')
logs_dir  = logs_root / 'logs'
csv_dir   = logs_root / 'csv'
rep_dir   = logs_root / 'reports'

print(f'Logs root : {logs_root}  (exists={logs_root.exists()})')
print(f'logs/     : {logs_dir.exists()}')
print(f'csv/      : {csv_dir.exists()}')
print(f'reports/  : {rep_dir.exists()}')

Logs root : /pipeline-logs  (exists=True)
logs/     : True
csv/      : True
reports/  : True


In [3]:
# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

MAX_LOG_LINES = 50  # max lines shown per text log


def show_log(path, max_lines=MAX_LOG_LINES):
    """Print the content of a text log file, or report if unavailable."""
    path = Path(path)
    print(f'\n=== {path} ===')
    if not path.exists():
        print('  (not available yet)')
        return
    if path.stat().st_size == 0:
        print('  (file exists but is empty)')
        return
    try:
        with path.open('r', encoding='utf-8', errors='replace') as fh:
            lines = fh.readlines()
        for line in lines[:max_lines]:
            print(line.rstrip('\n'))
        if len(lines) > max_lines:
            print(f'... ({len(lines) - max_lines} more lines — increase max_lines to see all)')
    except Exception as exc:
        print(f'  Could not read: {exc}')


def load_csv(path, label=None):
    """Load a CSV as a DataFrame and print a short summary."""
    path = Path(path)
    tag = label or path.name
    if not path.exists():
        print(f'[{tag}] not available yet.')
        return pd.DataFrame()
    if path.stat().st_size == 0:
        print(f'[{tag}] exists but is empty.')
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f'[{tag}] {len(df):,} rows × {len(df.columns)} columns')
        return df
    except Exception as exc:
        print(f'[{tag}] could not be loaded: {exc}')
        return pd.DataFrame()


def list_dir(d, indent=2):
    """Print a sorted listing of *d*."""
    d = Path(d)
    if not d.exists():
        print(' ' * indent + '(missing)')
        return
    items = sorted(d.iterdir(), key=lambda p: p.name.lower())
    if not items:
        print(' ' * indent + '(empty)')
        return
    for p in items:
        kind = 'DIR ' if p.is_dir() else 'FILE'
        size = '' if p.is_dir() else f'  {p.stat().st_size:>12,} B'
        print(' ' * indent + f'[{kind}] {p.name}{size}')

In [4]:
# Full recursive listing of /pipeline-logs
print('── pipeline-logs/')
for sub in ('logs', 'csv', 'reports'):
    print(f'   ── {sub}/')
    list_dir(logs_root / sub, indent=6)

── pipeline-logs/
   ── logs/
      [FILE] create_host_mapping.log         2,642 B
      [FILE] create_host_status_report.log         1,104 B
      [FILE] host_download.log    12,060,402 B
      [FILE] host_fasta_qc.csv       539,965 B
      [FILE] index_individual_host_sequences.log           276 B
      [FILE] index_phage_sequences.log         4,371 B
      [FILE] index_protein_sequences.log        69,097 B
      [FILE] merge_phage_fasta.log           691 B
      [FILE] merge_protein_fasta.log           697 B
   ── csv/
      [FILE] .host_indexes_complete             0 B
      [FILE] assembly_metadata.csv     1,342,345 B
      [FILE] host_metadata.csv       755,800 B
      [FILE] host_token_resolution_cache.json     3,676,198 B
      [FILE] phage_host_assemblies.csv   216,951,957 B
      [FILE] phage_host_candidates.csv   129,078,999 B
      [FILE] phage_host_links.csv   157,803,669 B
      [FILE] pipeline_run_provenance.csv           312 B
      [FILE] pipeline_run_provenance.json  

---
<a id='2-metadata'></a>
## 2 — Metadata Downloads & Merging

The pipeline downloads TSVs from PhageScope for each feature and merges them.
A ydata-profiling HTML report is generated per feature.  
No `.log` file is produced at this stage (downloads use `wget` directly via Snakemake shell rules).

> Files live in: `pipeline-logs/reports/{feature}_report.html`

In [5]:
FEATURES = [
    'phage_metadata',
    'annotated_proteins_metadata',
    'transcription_terminator_metadata',
    'phage_trna_tmrna_metadata',
    'phage_anti_crispr_metadata',
    'phage_virulent_factor_metadata',
    'phage_transmembrane_protein_metadata',
    'antimicrobial_resistance_gene_metadata',
    'crispr_array_metadata',
]

print('Metadata report availability:')
for feat in FEATURES:
    p = rep_dir / f'{feat}_report.html'
    status = f'✅  {p.stat().st_size:,} B' if p.exists() else '❌  missing'
    print(f'  {feat:<50} {status}')

Metadata report availability:
  phage_metadata                                     ✅  2,887,595 B
  annotated_proteins_metadata                        ✅  9,694,270 B
  transcription_terminator_metadata                  ✅  1,899,343 B
  phage_trna_tmrna_metadata                          ✅  2,620,070 B
  phage_anti_crispr_metadata                         ✅  1,268,195 B
  phage_virulent_factor_metadata                     ✅  1,726,834 B
  phage_transmembrane_protein_metadata               ✅  9,111,469 B
  antimicrobial_resistance_gene_metadata             ✅  1,611,985 B
  crispr_array_metadata                              ✅  16,378,720 B


The HTML reports can be opened directly in the browser (they are mounted at `./pipeline_logs/reports/`).

---
<a id='3-database'></a>
## 3 — Database Creation & Validation

Rules: `create_duckdb` → `optimize_database` → `validate_database`  
The validation step produces an HTML report summarising table counts and integrity checks.

> File: `pipeline-logs/reports/database_validation.html`

In [6]:
db_report = rep_dir / 'database_validation.html'
if db_report.exists():
    print(f'✅  database_validation.html  ({db_report.stat().st_size:,} B)')
    print('    Open in a browser or use the JupyterLab HTML viewer.')
else:
    print('❌  database_validation.html not available yet.')

✅  database_validation.html  (36,113 B)
    Open in a browser or use the JupyterLab HTML viewer.


---
<a id='4-host-download'></a>
## 4 — Host Resolution & Download

Rule: `download_host_genomes` (script: `download_host_genomes_robust.py`)

This step:
1. Parses each phage's `Host` field into candidate tokens.
2. Resolves tokens to NCBI assembly accessions via `AssemblyResolver`.
3. Downloads genome FASTA files via FTP.

**Log files**
| File | Description |
|------|-------------|
| `logs/host_download.log` | Full execution log (resolver + downloader) |
| `logs/host_download_failures.log` | Per-accession failures only |

**CSV outputs** (in `csv/`)
| File | Description |
|------|-------------|
| `phage_host_candidates.csv` | One row per (Phage_ID, Host_Token) |
| `phage_host_assemblies.csv` | Resolved assembly links with confidence scoring |
| `assembly_metadata.csv` | Download status per assembly |
| `host_metadata.csv` | High-level host metadata |
| `phage_host_links.csv` | Final phage → assembly links |
| `host_token_resolution_cache.json` | Persistent token→assembly cache |

In [7]:
# ── Text logs ──────────────────────────────────────────────────────────────
show_log(logs_dir / 'host_download.log')
show_log(logs_dir / 'host_download_failures.log')


=== /pipeline-logs/logs/host_download.log ===
2026-07-04 15:52:12,589 - INFO - ✅ AssemblyResolver initialized
2026-07-04 15:52:12,589 - INFO -    Email: th.schow@gmail.com
2026-07-04 15:52:12,589 - INFO -    API key: Yes
2026-07-04 15:52:12,589 - INFO -    Delay: 0.1s
2026-07-04 15:52:12,590 - INFO - ✅ RobustHostGenomeDownloader initialized
2026-07-04 15:52:12,590 - INFO -    Phage CSV: /data/intermediate/csv/merged/merged_phage_metadata.csv
2026-07-04 15:52:12,590 - INFO -    Output directory: /data/intermediate/fasta/hosts
2026-07-04 15:52:12,590 - INFO -    Metadata only: False
2026-07-04 15:52:12,590 - INFO -    Skip existing: True
2026-07-04 15:52:12,590 - INFO -    Validate checksums: True
2026-07-04 15:52:12,590 - INFO -    Reuse resolution cache: True
2026-07-04 15:52:12,591 - INFO -    Resolution cache file: /pipeline-logs/csv/host_token_resolution_cache.json
2026-07-04 15:52:12,591 - INFO - ================================================================================
2026

In [8]:
# ── phage_host_candidates.csv ──────────────────────────────────────────────
candidates = load_csv(csv_dir / 'phage_host_candidates.csv')
if not candidates.empty:
    print('\nColumns:', candidates.columns.tolist())
    display(candidates.head(10))
    
    print('\n--- Token type distribution ---')
    if 'Token_Type' in candidates.columns:
        display(candidates['Token_Type'].value_counts().rename('count'))
    
    print(f'\nUnique phages : {candidates["Phage_ID"].nunique():,}')
    print(f'Total tokens  : {len(candidates):,}')

[phage_host_candidates.csv] 1,356,533 rows × 5 columns

Columns: ['Phage_ID', 'Host_Raw', 'Host_Token', 'Token_Type', 'Token_Order']


,Phage_ID,Host_Raw,Host_Token,Token_Type,Token_Order
0,1111525849475051:k141_1295821_length_55304_cov_50.0000,Xylella fastidiosa,Xylella fastidiosa,species_name,1
1,1111525849475051:k141_659940_length_33502_cov_36.9752,Corynebacterium xerosis,Corynebacterium xerosis,species_name,1
2,1111525849475064:k141_1333017_length_3534_cov_29.0000,Salmonella enterica,Salmonella enterica,species_name,1
3,1111525849475830:k141_1051363_length_7863_cov_530.0000,Salmonella enterica,Salmonella enterica,species_name,1
4,1111525849475830:k141_1337912_length_7654_cov_66.0000,Bacillus cereus,Bacillus cereus,species_name,1
5,1111525849475830:k141_1720558_length_67458_cov_45.5609,Parabacteroides distasonis,Parabacteroides distasonis,species_name,1
6,1111525849475830:k141_1916821_length_3751_cov_51.0000,Dorea longicatena,Dorea longicatena,species_name,1
7,1111525849475830:k141_2058922_length_7592_cov_133.0178,Bacillus cereus,Bacillus cereus,species_name,1
8,1111525849475830:k141_2213952_length_7409_cov_41.0056,Salmonella enterica,Salmonella enterica,species_name,1
9,1111525849475830:k141_227495_length_7588_cov_936.0000,Salmonella enterica,Salmonella enterica,species_name,1



--- Token type distribution ---


Token_Type
species_name          1021163
other                  327766
assembly_accession       7604
Name: count, dtype: int64


Unique phages : 1,310,154
Total tokens  : 1,356,533


In [9]:
# ── phage_host_assemblies.csv ──────────────────────────────────────────────
assemblies = load_csv(csv_dir / 'phage_host_assemblies.csv')
if not assemblies.empty:
    print('\nColumns:', assemblies.columns.tolist())
    display(assemblies.head(10))

    if 'Resolution_Source' in assemblies.columns:
        print('\n--- Resolution source distribution ---')
        display(assemblies['Resolution_Source'].value_counts().rename('count'))

    if 'Ambiguous' in assemblies.columns:
        n_amb = assemblies['Ambiguous'].sum()
        print(f'\nAmbiguous resolutions : {n_amb:,} / {len(assemblies):,}')

    if 'Assembly_Level' in assemblies.columns:
        print('\n--- Assembly level distribution ---')
        display(assemblies['Assembly_Level'].value_counts().rename('count'))

[phage_host_assemblies.csv] 1,249,401 rows × 14 columns

Columns: ['Phage_ID', 'Host_Raw', 'Host_Token', 'Token_Type', 'Token_Order', 'Assembly_Accession', 'Resolution_Source', 'Resolution_Rank', 'Confidence', 'Assembly_Level', 'RefSeq_Category', 'Quality_Score', 'Ambiguous', 'Ambiguity_Reason']


,Phage_ID,Host_Raw,Host_Token,Token_Type,Token_Order,Assembly_Accession,Resolution_Source,Resolution_Rank,Confidence,Assembly_Level,RefSeq_Category,Quality_Score,Ambiguous,Ambiguity_Reason
0,1111525849475051:k141_1295821_length_55304_cov_50.0000,Xylella fastidiosa,Xylella fastidiosa,species_name,1,GCF_055959565.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
1,1111525849475051:k141_659940_length_33502_cov_36.9752,Corynebacterium xerosis,Corynebacterium xerosis,species_name,1,GCF_055583525.1,species_to_taxid_to_assembly,1,0.7,Scaffold,na,110,False,NaN
2,1111525849475064:k141_1333017_length_3534_cov_29.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
3,1111525849475830:k141_1051363_length_7863_cov_530.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
4,1111525849475830:k141_1337912_length_7654_cov_66.0000,Bacillus cereus,Bacillus cereus,species_name,1,GCF_058449145.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
5,1111525849475830:k141_1720558_length_67458_cov_45.5609,Parabacteroides distasonis,Parabacteroides distasonis,species_name,1,GCF_051904065.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
6,1111525849475830:k141_1916821_length_3751_cov_51.0000,Dorea longicatena,Dorea longicatena,species_name,1,GCF_042854315.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
7,1111525849475830:k141_2058922_length_7592_cov_133.0178,Bacillus cereus,Bacillus cereus,species_name,1,GCF_058449145.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
8,1111525849475830:k141_2213952_length_7409_cov_41.0056,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN
9,1111525849475830:k141_227495_length_7588_cov_936.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1,0.7,Complete Genome,na,1010,False,NaN



--- Resolution source distribution ---


Resolution_Source
species_to_taxid_to_assembly    961249
fallback                        280548
accession_in_host_field           7604
Name: count, dtype: int64


Ambiguous resolutions : 0 / 1,249,401

--- Assembly level distribution ---


Assembly_Level
Complete Genome    1066719
Scaffold            102939
Contig               62535
Chromosome           17208
Name: count, dtype: int64

In [10]:
# ── assembly_metadata.csv ─────────────────────────────────────────────────
asm_meta = load_csv(csv_dir / 'assembly_metadata.csv')
if not asm_meta.empty:
    print('\nColumns:', asm_meta.columns.tolist())
    display(asm_meta.head(10))

    if 'Download_Status' in asm_meta.columns:
        print('\n--- Download status distribution ---')
        display(asm_meta['Download_Status'].value_counts().rename('count'))

    if 'Metadata_Only' in asm_meta.columns:
        print(f'\nMetadata-only (no FASTA downloaded): {asm_meta["Metadata_Only"].sum():,}')

[assembly_metadata.csv] 5,524 rows × 17 columns

Columns: ['Assembly_Accession', 'Assembly_Name', 'Organism_Name', 'Species_TaxID', 'Strain', 'Assembly_Level', 'RefSeq_Category', 'BioSample', 'BioProject', 'FTP_Path', 'Submission_Date', 'Is_Latest', 'Quality_Score', 'Is_RefSeq', 'Download_Status', 'Download_Date', 'Metadata_Only']


,Assembly_Accession,Assembly_Name,Organism_Name,Species_TaxID,Strain,Assembly_Level,RefSeq_Category,BioSample,BioProject,FTP_Path,Submission_Date,Is_Latest,Quality_Score,Is_RefSeq,Download_Status,Download_Date,Metadata_Only
0,GCF_055959565.1,ASM5595956v1,Xylella fastidiosa,2371,-,Complete Genome,na,SAMN46769223,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/959/565/GCF_055959565.1_ASM55...,2026/03/18 00:00,True,1010,True,success,2026-07-04,False
1,GCF_055583525.1,ASM5558352v1,Corynebacterium xerosis,1725,-,Scaffold,na,SAMN55302120,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/583/525/GCF_055583525.1_ASM55...,2026/03/01 00:00,True,110,True,success,2026-07-04,False
2,GCF_058500565.1,ASM5850056v1,Salmonella enterica,28901,-,Complete Genome,na,SAMN43257597,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/058/500/565/GCF_058500565.1_ASM58...,2026/06/29 00:00,True,1010,True,success,2026-07-04,False
3,GCF_058449145.1,ASM5844914v1,Bacillus cereus,1396,-,Complete Genome,na,SAMN47628865,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/058/449/145/GCF_058449145.1_ASM58...,2026/06/28 00:00,True,1010,True,success,2026-07-04,False
4,GCF_051904065.1,ASM5190406v1,Parabacteroides distasonis,823,-,Complete Genome,na,SAMN40870503,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/051/904/065/GCF_051904065.1_ASM51...,2025/08/10 00:00,True,1010,True,success,2026-07-04,False
5,GCF_042854315.1,ASM4285431v1,Dorea longicatena,88431,-,Complete Genome,na,SAMD00759010,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/042/854/315/GCF_042854315.1_ASM42...,2024/10/13 00:00,True,1010,True,success,2026-07-04,False
6,GCF_058693925.1,ASM5869392v1,Escherichia coli,562,-,Complete Genome,na,SAMN47596159,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/058/693/925/GCA_058693925.1_ASM58...,2026/07/03 00:00,True,1010,True,failed,2026-07-04,False
7,GCF_058344415.1,ASM5834441v1,Staphylococcus epidermidis,1282,-,Complete Genome,na,SAMN36735441,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/058/344/415/GCF_058344415.1_ASM58...,2026/06/25 00:00,True,1010,True,success,2026-07-04,False
8,GCF_039773935.1,S260,Corynebacterium striatum,43770,-,Chromosome,na,SAMN34403534,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/039/773/935/GCF_039773935.1_S260,2024/05/24 00:00,True,510,True,success,2026-07-04,False
9,GCF_039728355.1,ASM3972835v1,Burkholderia cenocepacia,95486,-,Complete Genome,na,SAMN41377768,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/039/728/355/GCF_039728355.1_ASM39...,2024/05/24 00:00,True,1010,True,success,2026-07-04,False



--- Download status distribution ---


Download_Status
success    5512
failed       12
Name: count, dtype: int64


Metadata-only (no FASTA downloaded): 0


In [11]:
# ── host_metadata.csv ────────────────────────────────────────────────────
host_meta = load_csv(csv_dir / 'host_metadata.csv')
if not host_meta.empty:
    print('\nColumns:', host_meta.columns.tolist())
    display(host_meta.head(10))

[host_metadata.csv] 5,524 rows × 11 columns

Columns: ['Host_ID', 'Species_Name', 'Strain_Name', 'Assembly_Accession', 'Assembly_Name', 'Assembly_Level', 'Genome_Length', 'GC_Content', 'RefSeq_Category', 'Download_Date', 'Source']


,Host_ID,Species_Name,Strain_Name,Assembly_Accession,Assembly_Name,Assembly_Level,Genome_Length,GC_Content,RefSeq_Category,Download_Date,Source
0,GCF_055959565_1,Xylella fastidiosa,-,GCF_055959565.1,ASM5595956v1,Complete Genome,2619904,52.09,na,2026-07-04,assembly_resolver
1,GCF_055583525_1,Corynebacterium xerosis,-,GCF_055583525.1,ASM5558352v1,Scaffold,2708450,69.6,na,2026-07-04,assembly_resolver
2,GCF_058500565_1,Salmonella enterica,-,GCF_058500565.1,ASM5850056v1,Complete Genome,4749163,52.28,na,2026-07-04,assembly_resolver
3,GCF_058449145_1,Bacillus cereus,-,GCF_058449145.1,ASM5844914v1,Complete Genome,5696470,35.13,na,2026-07-04,assembly_resolver
4,GCF_051904065_1,Parabacteroides distasonis,-,GCF_051904065.1,ASM5190406v1,Complete Genome,5163018,45.09,na,2026-07-04,assembly_resolver
5,GCF_042854315_1,Dorea longicatena,-,GCF_042854315.1,ASM4285431v1,Complete Genome,3179541,41.24,na,2026-07-04,assembly_resolver
6,GCF_058693925_1,Escherichia coli,-,GCF_058693925.1,ASM5869392v1,Complete Genome,-,-,na,2026-07-04,assembly_resolver
7,GCF_058344415_1,Staphylococcus epidermidis,-,GCF_058344415.1,ASM5834441v1,Complete Genome,2425999,32.21,na,2026-07-04,assembly_resolver
8,GCF_039773935_1,Corynebacterium striatum,-,GCF_039773935.1,S260,Chromosome,2804500,58.91,na,2026-07-04,assembly_resolver
9,GCF_039728355_1,Burkholderia cenocepacia,-,GCF_039728355.1,ASM3972835v1,Complete Genome,8270568,66.71,na,2026-07-04,assembly_resolver


In [12]:
# ── phage_host_links.csv ─────────────────────────────────────────────────
links = load_csv(csv_dir / 'phage_host_links.csv')
if not links.empty:
    print('\nColumns:', links.columns.tolist())
    display(links.head(10))
    print(f'\nUnique phages with a host link : {links["Phage_ID"].nunique():,}')

[phage_host_links.csv] 1,241,293 rows × 7 columns

Columns: ['Phage_ID', 'Host_Species', 'Host_Full_Name', 'Assembly_Accession', 'Assembly_Level', 'RefSeq_Category', 'Link_Quality']


,Phage_ID,Host_Species,Host_Full_Name,Assembly_Accession,Assembly_Level,RefSeq_Category,Link_Quality
0,1111525849475051:k141_1295821_length_55304_cov_50.0000,Xylella fastidiosa,Xylella fastidiosa,GCF_055959565.1,Complete Genome,na,direct
1,1111525849475051:k141_659940_length_33502_cov_36.9752,Corynebacterium xerosis,Corynebacterium xerosis,GCF_055583525.1,Scaffold,na,direct
2,1111525849475064:k141_1333017_length_3534_cov_29.0000,Salmonella enterica,Salmonella enterica,GCF_058500565.1,Complete Genome,na,direct
3,1111525849475830:k141_1051363_length_7863_cov_530.0000,Salmonella enterica,Salmonella enterica,GCF_058500565.1,Complete Genome,na,direct
4,1111525849475830:k141_1337912_length_7654_cov_66.0000,Bacillus cereus,Bacillus cereus,GCF_058449145.1,Complete Genome,na,direct
5,1111525849475830:k141_1720558_length_67458_cov_45.5609,Parabacteroides distasonis,Parabacteroides distasonis,GCF_051904065.1,Complete Genome,na,direct
6,1111525849475830:k141_1916821_length_3751_cov_51.0000,Dorea longicatena,Dorea longicatena,GCF_042854315.1,Complete Genome,na,direct
7,1111525849475830:k141_2058922_length_7592_cov_133.0178,Bacillus cereus,Bacillus cereus,GCF_058449145.1,Complete Genome,na,direct
8,1111525849475830:k141_2213952_length_7409_cov_41.0056,Salmonella enterica,Salmonella enterica,GCF_058500565.1,Complete Genome,na,direct
9,1111525849475830:k141_227495_length_7588_cov_936.0000,Salmonella enterica,Salmonella enterica,GCF_058500565.1,Complete Genome,na,direct



Unique phages with a host link : 1,218,983


In [13]:
# ── host_token_resolution_cache.json ────────────────────────────────────
cache_path = csv_dir / 'host_token_resolution_cache.json'
if cache_path.exists():
    with cache_path.open('r') as fh:
        cache = json.load(fh)
    n_tokens = len(cache)
    n_resolved = sum(1 for v in cache.values() if v)
    n_empty = n_tokens - n_resolved
    print(f'Token resolution cache: {n_tokens:,} tokens total')
    print(f'  Resolved (≥1 assembly) : {n_resolved:,}')
    print(f'  Empty (no assembly)    : {n_empty:,}')
else:
    print('host_token_resolution_cache.json not available yet.')

Token resolution cache: 10,778 tokens total
  Resolved (≥1 assembly) : 6,207
  Empty (no assembly)    : 4,571


---
<a id='5-host-mapping'></a>
## 5 — Host Mapping Creation

Rule: `create_host_mapping` (script: `create_host_mapping.py`)

Builds a `Host_ID → FASTA path` JSON from the downloaded assemblies and any private host data.

> Log file: `logs/create_host_mapping.log`

In [14]:
show_log(logs_dir / 'create_host_mapping.log')


=== /pipeline-logs/logs/create_host_mapping.log ===
2026-07-05 03:53:44,245 - INFO - ======================================================================
2026-07-05 03:53:44,245 - INFO - 🚀 Starting host mapping creation
2026-07-05 03:53:44,245 - INFO -    Metadata CSV    : /pipeline-logs/csv/host_metadata.csv
2026-07-05 03:53:44,245 - INFO -    Private mapping : /private-data/private_host_fasta_mapping.json
2026-07-05 03:53:44,245 - INFO -    Input dir       : /data/intermediate/fasta/hosts
2026-07-05 03:53:44,245 - INFO -    Output          : /data/processed/sequences/host_fasta_mapping.json
2026-07-05 03:53:44,274 - INFO -    Host rows in CSV: 5524
2026-07-05 03:53:44,276 - WARNING - ⚠️  Missing or empty file: /data/intermediate/fasta/hosts/GCF_058693925_1.fna
2026-07-05 03:53:44,375 - WARNING - ⚠️  Missing or empty file: /data/intermediate/fasta/hosts/GCF_986126175_1.fna
2026-07-05 03:53:44,452 - WARNING - ⚠️  Missing or empty file: /data/intermediate/fasta/hosts/GCF_058342895_1.

---
<a id='6-host-qc'></a>
## 6 — Host FASTA QC & Indexing

Rule: `index_individual_host_sequences` (script: `index_individual_hosts.py`)

QC procedure per host FASTA (non-destructive):
1. **Header audit**: files with duplicate sequence identifiers are *rejected* (not indexed).
2. **Sequence content audit**: identical sequences trigger a warning but the file is still indexed.

**Outputs**
| File | Description |
|------|-------------|
| `logs/index_individual_host_sequences.log` | Step-by-step execution log |
| `logs/host_fasta_qc.csv` | Per-file QC results (DataFrame-loadable) |

In [15]:
show_log(logs_dir / 'index_individual_host_sequences.log')


=== /pipeline-logs/logs/index_individual_host_sequences.log ===
Total host FASTA files: 5517
Newly indexed: 5512
Already indexed (skipped): 5
Rejected (duplicate headers): 0
Errors: 0

QC log: /pipeline-logs/logs/host_fasta_qc.csv
2026-07-05 04:03:04,766 - INFO - Indexing complete: 5512 newly indexed, 5 pre-existing, 0 rejected, 0 errors


In [16]:
# ── host_fasta_qc.csv ─────────────────────────────────────────────────────
qc = load_csv(logs_dir / 'host_fasta_qc.csv')
if not qc.empty:
    print('\nColumns:', qc.columns.tolist())
    display(qc.head(10))

    if 'index_status' in qc.columns:
        print('\n--- Index status distribution ---')
        display(qc['index_status'].value_counts().rename('count'))

    if 'header_qc_status' in qc.columns:
        print('\n--- Header QC status distribution ---')
        display(qc['header_qc_status'].value_counts().rename('count'))

    if 'seq_qc_status' in qc.columns:
        print('\n--- Sequence QC status distribution ---')
        display(qc['seq_qc_status'].value_counts().rename('count'))

    # Rejected files (duplicate headers)
    if 'index_status' in qc.columns:
        rejected = qc[qc['index_status'] == 'rejected_duplicate_headers']
        if not rejected.empty:
            print(f'\n--- Rejected files ({len(rejected):,}) ---')
            display(rejected[['host_id', 'n_duplicate_headers',
                               'duplicate_header_examples', 'error_message']].head(20))

    # Files with duplicate sequences (warning)
    if 'n_identical_seq_groups' in qc.columns:
        dup_seq = qc[qc['n_identical_seq_groups'].fillna(0) > 0]
        if not dup_seq.empty:
            print(f'\n--- Files with duplicate sequences ({len(dup_seq):,}) ---')
            display(dup_seq[['host_id', 'n_identical_seq_groups',
                              'identical_seq_group_examples']].head(20))

[host_fasta_qc.csv] 5,517 rows × 12 columns

Columns: ['host_id', 'fasta_path', 'index_existed', 'total_sequences', 'header_qc_status', 'n_duplicate_headers', 'duplicate_header_examples', 'seq_qc_status', 'n_identical_seq_groups', 'identical_seq_group_examples', 'index_status', 'error_message']


,host_id,fasta_path,index_existed,total_sequences,header_qc_status,n_duplicate_headers,duplicate_header_examples,seq_qc_status,n_identical_seq_groups,identical_seq_group_examples,index_status,error_message
0,GCF_055959565_1,/data/intermediate/fasta/hosts/GCF_055959565_1.fna,False,1,ok,0,NaN,ok,0,NaN,indexed,NaN
1,GCF_055583525_1,/data/intermediate/fasta/hosts/GCF_055583525_1.fna,False,8,ok,0,NaN,ok,0,NaN,indexed,NaN
2,GCF_058500565_1,/data/intermediate/fasta/hosts/GCF_058500565_1.fna,False,4,ok,0,NaN,ok,0,NaN,indexed,NaN
3,GCF_058449145_1,/data/intermediate/fasta/hosts/GCF_058449145_1.fna,False,5,ok,0,NaN,ok,0,NaN,indexed,NaN
4,GCF_051904065_1,/data/intermediate/fasta/hosts/GCF_051904065_1.fna,False,2,ok,0,NaN,ok,0,NaN,indexed,NaN
5,GCF_042854315_1,/data/intermediate/fasta/hosts/GCF_042854315_1.fna,False,1,ok,0,NaN,ok,0,NaN,indexed,NaN
6,GCF_058344415_1,/data/intermediate/fasta/hosts/GCF_058344415_1.fna,False,1,ok,0,NaN,ok,0,NaN,indexed,NaN
7,GCF_039773935_1,/data/intermediate/fasta/hosts/GCF_039773935_1.fna,False,1,ok,0,NaN,ok,0,NaN,indexed,NaN
8,GCF_039728355_1,/data/intermediate/fasta/hosts/GCF_039728355_1.fna,False,5,ok,0,NaN,ok,0,NaN,indexed,NaN
9,GCF_057457685_1,/data/intermediate/fasta/hosts/GCF_057457685_1.fna,False,5,ok,0,NaN,ok,0,NaN,indexed,NaN



--- Index status distribution ---


index_status
indexed            5512
already_indexed       5
Name: count, dtype: int64


--- Header QC status distribution ---


header_qc_status
ok    5517
Name: count, dtype: int64


--- Sequence QC status distribution ---


seq_qc_status
ok                             5514
warning_identical_sequences       3
Name: count, dtype: int64


--- Files with duplicate sequences (3) ---


,host_id,n_identical_seq_groups,identical_seq_group_examples
641,GCF_000245735_1,2,"NZ_AHTH01000061.1 Alishewanella jeotgali KCTC 22429 contig061, whole genome ..."
1697,GCF_001046895_1,1,"NZ_CAQI01000016.1 Pseudarthrobacter siccitolerans strain 4J27, whole genome ..."
1915,GCF_001431085_1,1,"NZ_BBRU01000548.1 Candidatus Symbiothrix dinenymphae strain C5-4h, whole gen..."


---
<a id='7-host-status'></a>
## 7 — Host Status Report

Rule: `create_host_status_report` (script: `create_host_status_report.py`)

Joins the four upstream tables (candidates, assemblies, assembly_metadata, QC log)
into a single per-`(Phage_ID, Host_Token)` summary.

**Outputs**
| File | Description |
|------|-------------|
| `logs/create_host_status_report.log` | Step-by-step log |
| `reports/host_status_report.csv` | Combined status table |

In [17]:
show_log(logs_dir / 'create_host_status_report.log')


=== /pipeline-logs/logs/create_host_status_report.log ===
2026-07-05 04:03:07,771 - INFO - 📋 Loading upstream tables…
2026-07-05 04:03:10,215 - INFO -    candidates: 1356533 rows
2026-07-05 04:03:13,885 - INFO -    assemblies: 1249401 rows
2026-07-05 04:03:13,915 - INFO -    assembly_metadata: 5524 rows
2026-07-05 04:03:13,929 - INFO -    qc_log: 5517 rows
2026-07-05 04:03:40,353 - INFO - ============================================================
2026-07-05 04:03:40,354 - INFO - HOST STATUS REPORT SUMMARY
2026-07-05 04:03:40,354 - INFO - ============================================================
2026-07-05 04:03:40,354 - INFO - Total phages:                    1310154
2026-07-05 04:03:40,354 - INFO - Phages with ≥1 resolved host:    1218983
2026-07-05 04:03:40,354 - INFO - Assemblies downloaded:             5512
2026-07-05 04:03:40,354 - INFO - Assemblies indexed:                5512
2026-07-05 04:03:40,354 - INFO - Assemblies rejected (dup hdrs):       0
2026-07-05 04:03:40,354 -

In [18]:
# ── host_status_report.csv ───────────────────────────────────────────────
status_report = load_csv(rep_dir / 'host_status_report.csv')
if not status_report.empty:
    print('\nColumns:', status_report.columns.tolist())
    display(status_report.head(10))

    # Key summary metrics
    n_phages  = status_report['Phage_ID'].nunique()
    print(f'\nTotal unique phages            : {n_phages:>8,}')

    if 'Resolution_Status' in status_report.columns:
        n_res = status_report.loc[
            status_report['Resolution_Status'] == 'resolved', 'Phage_ID'
        ].nunique()
        print(f'Phages with ≥1 resolved host   : {n_res:>8,}')
        print(f'Phages with no resolved host   : {n_phages - n_res:>8,}')

        print('\n--- Resolution status distribution ---')
        display(status_report['Resolution_Status'].value_counts().rename('rows'))

    if 'Download_Status' in status_report.columns:
        print('\n--- Download status distribution ---')
        display(status_report['Download_Status'].value_counts(dropna=False).rename('rows'))

    if 'index_status' in status_report.columns:
        print('\n--- Index status distribution ---')
        display(status_report['index_status'].value_counts(dropna=False).rename('rows'))

    # Assemblies with failed downloads
    if 'Download_Status' in status_report.columns:
        failures = status_report[
            status_report['Download_Status'].notna() &
            (status_report['Download_Status'] != 'success')
        ].drop_duplicates('Assembly_Accession')
        if not failures.empty:
            cols = [c for c in ['Assembly_Accession', 'Organism_Name', 'Strain',
                                 'Download_Status'] if c in failures.columns]
            print(f'\n--- Download failures ({len(failures):,} assemblies) ---')
            display(failures[cols].head(20))

[host_status_report.csv] 1,356,533 rows × 29 columns

Columns: ['Phage_ID', 'Host_Raw', 'Host_Token', 'Token_Type', 'Token_Order', 'Assembly_Accession', 'Resolution_Source', 'Resolution_Rank', 'Confidence', 'Assembly_Level', 'RefSeq_Category', 'Quality_Score', 'Ambiguous', 'Ambiguity_Reason', 'Resolution_Status', 'Download_Status', 'Download_Date', 'Metadata_Only', 'Organism_Name', 'Strain', 'total_sequences', 'header_qc_status', 'n_duplicate_headers', 'duplicate_header_examples', 'seq_qc_status', 'n_identical_seq_groups', 'identical_seq_group_examples', 'index_status', 'error_message']


,Phage_ID,Host_Raw,Host_Token,Token_Type,Token_Order,Assembly_Accession,Resolution_Source,Resolution_Rank,Confidence,Assembly_Level,RefSeq_Category,Quality_Score,Ambiguous,Ambiguity_Reason,Resolution_Status,Download_Status,Download_Date,Metadata_Only,Organism_Name,Strain,total_sequences,header_qc_status,n_duplicate_headers,duplicate_header_examples,seq_qc_status,n_identical_seq_groups,identical_seq_group_examples,index_status,error_message
0,1111525849475051:k141_1295821_length_55304_cov_50.0000,Xylella fastidiosa,Xylella fastidiosa,species_name,1,GCF_055959565.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Xylella fastidiosa,-,1.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
1,1111525849475051:k141_659940_length_33502_cov_36.9752,Corynebacterium xerosis,Corynebacterium xerosis,species_name,1,GCF_055583525.1,species_to_taxid_to_assembly,1.0,0.7,Scaffold,na,110.0,False,NaN,resolved,success,2026-07-04,False,Corynebacterium xerosis,-,8.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
2,1111525849475064:k141_1333017_length_3534_cov_29.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Salmonella enterica,-,4.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
3,1111525849475830:k141_1051363_length_7863_cov_530.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Salmonella enterica,-,4.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
4,1111525849475830:k141_1337912_length_7654_cov_66.0000,Bacillus cereus,Bacillus cereus,species_name,1,GCF_058449145.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Bacillus cereus,-,5.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
5,1111525849475830:k141_1720558_length_67458_cov_45.5609,Parabacteroides distasonis,Parabacteroides distasonis,species_name,1,GCF_051904065.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Parabacteroides distasonis,-,2.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
6,1111525849475830:k141_1916821_length_3751_cov_51.0000,Dorea longicatena,Dorea longicatena,species_name,1,GCF_042854315.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Dorea longicatena,-,1.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
7,1111525849475830:k141_2058922_length_7592_cov_133.0178,Bacillus cereus,Bacillus cereus,species_name,1,GCF_058449145.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Bacillus cereus,-,5.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
8,1111525849475830:k141_2213952_length_7409_cov_41.0056,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Salmonella enterica,-,4.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN
9,1111525849475830:k141_227495_length_7588_cov_936.0000,Salmonella enterica,Salmonella enterica,species_name,1,GCF_058500565.1,species_to_taxid_to_assembly,1.0,0.7,Complete Genome,na,1010.0,False,NaN,resolved,success,2026-07-04,False,Salmonella enterica,-,4.0,ok,0.0,NaN,ok,0.0,NaN,indexed,NaN



Total unique phages            : 1,310,154
Phages with ≥1 resolved host   : 1,218,983
Phages with no resolved host   :   91,171

--- Resolution status distribution ---


Resolution_Status
resolved      1249401
unresolved     107132
Name: rows, dtype: int64


--- Download status distribution ---


Download_Status
success    1197537
NaN         107132
failed       51864
Name: rows, dtype: int64


--- Index status distribution ---


index_status
indexed    1197537
NaN         158996
Name: rows, dtype: int64


--- Download failures (12 assemblies) ---


,Assembly_Accession,Organism_Name,Strain,Download_Status
11,GCF_058693925.1,Escherichia coli,-,failed
126002,GCF_986126175.1,Dialister succinatiphilus,-,failed
139182,GCF_058342895.1,Marinobacter salarius,-,failed
163390,GCF_058342905.1,Alloalcanivorax xenomutans,-,failed
224701,GCF_985670335.1,Helcococcus sp. SA-HE-C1,-,failed
739257,GCF_986141305.1,Streptococcus sp.,-,failed
1042227,GCF_986114505.1,Alistipes sp.,-,failed
1064483,GCF_986095795.1,Veillonella sp.,-,failed
1072626,GCF_986109425.1,Bifidobacterium sp.,-,failed
1189232,GCA_002491635.1,Bacteroides sp. UBA7156,-,failed


---
<a id='8-fasta-merge'></a>
## 8 — Phage & Protein FASTA Merging

Rules: `merge_phage_fasta` and `merge_protein_fasta` (script: `merge_fasta.py`)

Merges per-source FASTA files into a single combined FASTA for phages and proteins.

> Log files: `logs/merge_phage_fasta.log`, `logs/merge_protein_fasta.log`

In [19]:
show_log(logs_dir / 'merge_phage_fasta.log')


=== /pipeline-logs/logs/merge_phage_fasta.log ===
2026-07-04 15:54:52,034 - INFO - ======================================================================
2026-07-04 15:54:52,034 - INFO - 🚀 Starting phage FASTA merge
2026-07-04 15:54:52,034 - INFO -    Expected inputs : 26
2026-07-04 15:54:52,035 - INFO -    Output          : /data/processed/sequences/all_phages.fasta
2026-07-04 15:54:52,035 - INFO -    Valid inputs    : 26/26
2026-07-04 15:59:16,175 - INFO - ✅ Merged 26/26 phage FASTA files
2026-07-04 15:59:16,176 - INFO -    Output size     : 53,998,656,984 bytes
2026-07-04 15:59:16,176 - INFO -    Elapsed         : 264.1s
2026-07-04 15:59:16,176 - INFO - ======================================================================


In [20]:
show_log(logs_dir / 'merge_protein_fasta.log')


=== /pipeline-logs/logs/merge_protein_fasta.log ===
2026-07-04 17:57:54,089 - INFO - ======================================================================
2026-07-04 17:57:54,090 - INFO - 🚀 Starting protein FASTA merge
2026-07-04 17:57:54,090 - INFO -    Expected inputs : 26
2026-07-04 17:57:54,090 - INFO -    Output          : /data/processed/sequences/all_proteins.fasta
2026-07-04 17:57:54,090 - INFO -    Valid inputs    : 26/26
2026-07-04 18:00:23,830 - INFO - ✅ Merged 26/26 protein FASTA files
2026-07-04 18:00:23,830 - INFO -    Output size     : 23,159,451,038 bytes
2026-07-04 18:00:23,830 - INFO -    Elapsed         : 149.7s
2026-07-04 18:00:23,830 - INFO - ======================================================================


---
<a id='9-fasta-index'></a>
## 9 — Phage & Protein FASTA Indexing

Rules: `index_phage_sequences` and `index_protein_sequences` (script: `index_sequences.py`)

Each step:
1. Normalises and deduplicates sequences in place.
2. Creates a pyfaidx `.fai` index.
3. Writes a duplicate analysis HTML report to `reports/`.

> Log files: `logs/index_phage_sequences.log`, `logs/index_protein_sequences.log`  
> HTML reports: `reports/{fasta_name}_duplicates.html`

In [21]:
show_log(logs_dir / 'index_phage_sequences.log')


=== /pipeline-logs/logs/index_phage_sequences.log ===
2026-07-04 15:59:20,370 - INFO - 🚀 Starting FASTA indexing
2026-07-04 15:59:20,370 - INFO - 📂 Input: /data/processed/sequences/all_phages.fasta
2026-07-04 15:59:20,370 - INFO - 📂 Output: /data/processed/sequences/all_phages.fasta.fai
2026-07-04 15:59:20,422 - INFO - 💾 Available memory: 337.0 GB / 377.8 GB
2026-07-04 15:59:20,422 - INFO - 📦 Input file size: 50.29 GB
2026-07-04 15:59:20,422 - INFO - 🔍 Validating basic FASTA format
2026-07-04 15:59:20,423 - INFO - 🔧 Step 1: Normalizing and deduplicating FASTA (in-place)
2026-07-04 15:59:20,423 - INFO - ⚠️  This will modify the original file: /data/processed/sequences/all_phages.fasta
2026-07-04 15:59:20,423 - INFO - 🔧 Normalizing and deduplicating FASTA: /data/processed/sequences/all_phages.fasta
2026-07-04 15:59:20,423 - INFO - 📋 Using FULL header line as sequence ID
2026-07-04 16:00:26,370 - INFO -    📊 Processed 100,000 sequences, kept 100,000...
2026-07-04 16:01:10,607 - INFO -   

In [22]:
show_log(logs_dir / 'index_protein_sequences.log')


=== /pipeline-logs/logs/index_protein_sequences.log ===
2026-07-04 18:00:27,174 - INFO - 🚀 Starting FASTA indexing
2026-07-04 18:00:27,175 - INFO - 📂 Input: /data/processed/sequences/all_proteins.fasta
2026-07-04 18:00:27,175 - INFO - 📂 Output: /data/processed/sequences/all_proteins.fasta.fai
2026-07-04 18:00:27,219 - INFO - 💾 Available memory: 357.6 GB / 377.8 GB
2026-07-04 18:00:27,220 - INFO - 📦 Input file size: 21.57 GB
2026-07-04 18:00:27,220 - INFO - 🔍 Validating basic FASTA format
2026-07-04 18:00:27,220 - INFO - 🔧 Step 1: Normalizing and deduplicating FASTA (in-place)
2026-07-04 18:00:27,220 - INFO - ⚠️  This will modify the original file: /data/processed/sequences/all_proteins.fasta
2026-07-04 18:00:27,220 - INFO - 🔧 Normalizing and deduplicating FASTA: /data/processed/sequences/all_proteins.fasta
2026-07-04 18:00:27,220 - INFO - 📋 Using FULL header line as sequence ID
2026-07-04 18:00:28,210 - INFO -    📊 Processed 100,000 sequences, kept 100,000...
2026-07-04 18:00:29,237 -

In [23]:
# Duplicate analysis HTML reports (generated by index_sequences.py)
dup_reports = sorted(rep_dir.glob('*_duplicates.html')) if rep_dir.exists() else []
if dup_reports:
    print('Duplicate analysis reports:')
    for p in dup_reports:
        print(f'  ✅  {p.name}  ({p.stat().st_size:,} B)')
else:
    print('No duplicate analysis reports found yet.')

Duplicate analysis reports:
  ✅  all_phages_duplicates.html  (349,955 B)


--- <a id='10-provenance-private'></a>
## 10 — Provenance and Private-Validation Diagnostics

This section helps diagnose source/provenance issues and missing private sources.


In [24]:
# ── Public provenance manifests ─────────────────────────────────────────────
pub_manifest_csv = load_csv(csv_dir / 'public_data_manifest.csv')
if not pub_manifest_csv.empty:
    cols = [c for c in ['feature', 'source_key', 'status', 'error_message', 'provider_release', 'provider_snapshot_date', 'provider_schema_profile'] if c in pub_manifest_csv.columns]
    display(pub_manifest_csv[cols].head(30))

    if 'status' in pub_manifest_csv.columns:
        print('Provenance status counts:')
        print(pub_manifest_csv['status'].value_counts(dropna=False))

        failed_rows = pub_manifest_csv[pub_manifest_csv['status'] != 'success']
        if len(failed_rows) > 0:
            print('[WARN] Non-success provenance rows:')
            display(failed_rows[cols])

run_prov_csv = load_csv(csv_dir / 'pipeline_run_provenance.csv')
if not run_prov_csv.empty:
    display(run_prov_csv.head(10))

# ── Private-source validation manifest ─────────────────────────────────────
private_manifest_candidates = [
    Path('/private-data/private_manifest.json'),
    Path.cwd().parent / 'private_data' / 'private_manifest.json',
    Path.cwd() / 'private_data' / 'private_manifest.json',
]
private_manifest = None
for candidate in private_manifest_candidates:
    if candidate.exists():
        private_manifest = candidate
        break

if private_manifest is None:
    print('[INFO] private_manifest.json not found in expected locations.')
else:
    print(f'Using private manifest: {private_manifest}')
    try:
        payload = json.loads(private_manifest.read_text())
    except json.JSONDecodeError as e:
        print(f"[WARN] Could not parse private manifest JSON: {e}")
        payload = {}

    print({k: payload.get(k) for k in ['sources_found', 'sources_valid', 'sources_invalid']})

    sources_df = pd.DataFrame(payload.get('sources', []))
    if not sources_df.empty:
        cols = [c for c in ['source_db', 'is_valid', 'source_dir', 'errors', 'warnings'] if c in sources_df.columns]
        display(sources_df[cols])



[public_data_manifest.csv] 229 rows × 18 columns


,feature,source_key,status,error_message,provider_release,provider_snapshot_date,provider_schema_profile
0,annotated_proteins_metadata,BAPS_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
1,annotated_proteins_metadata,CHVD_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
2,annotated_proteins_metadata,DDBJ_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
3,annotated_proteins_metadata,ELGV_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
4,annotated_proteins_metadata,EMBL_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
5,annotated_proteins_metadata,GOV2_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
6,annotated_proteins_metadata,GPD_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
7,annotated_proteins_metadata,GSV_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
8,annotated_proteins_metadata,GVD_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1
9,annotated_proteins_metadata,Genbank_Annotated_Proteins_metadata_URL,success,NaN,rolling,2026-05-11,phagescope_v1


Provenance status counts:
status
success    216
failed      13
Name: count, dtype: int64
[WARN] Non-success provenance rows:


,feature,source_key,status,error_message,provider_release,provider_snapshot_date,provider_schema_profile
43,antimicrobial_resistance_gene_metadata,TYMEFLIES_Antimicrobial_Resistance_Gene_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
96,phage_anti_crispr_metadata,TemPhD_Phage_AntiCRISPR_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
126,phage_transmembrane_protein_metadata,BAPS_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
129,phage_transmembrane_protein_metadata,ELGV_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
140,phage_transmembrane_protein_metadata,MetaVR_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
141,phage_transmembrane_protein_metadata,OPD_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
142,phage_transmembrane_protein_metadata,OVD_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
146,phage_transmembrane_protein_metadata,SVD_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
147,phage_transmembrane_protein_metadata,TYMEFLIES_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1
151,phage_transmembrane_protein_metadata,VMGC_Phage_Transmembrane_Protein_Metadata_URL,failed,HTTP Error 500: Internal Server Error,rolling,2026-05-11,phagescope_v1


[pipeline_run_provenance.csv] 1 rows × 10 columns


,pipeline_run_timestamp,provider_name,provider_release,provider_snapshot_date,provider_schema_profile,provider_api_base_url,provider_provenance_mode,pbi_version,git_commit,download_records_count
0,2026-07-04T15:56:50Z,PhageScope,rolling,2026-05-11,phagescope_v1,https://phageapi.deepomics.org,config_pinned,NaN,NaN,229


Using private manifest: /private-data/private_manifest.json
{'sources_found': 2, 'sources_valid': 2, 'sources_invalid': 0}


,source_db,is_valid,source_dir,errors,warnings
0,test_private,True,/private-data/test_private,[],[]
1,test_private_2,True,/private-data/test_private_2,[],[]
